# LightGBM với 13 đặc trưng động và nhãn khởi phát v1

Upload `smoke_detection_iot_transient_v1.csv` thành Kaggle Dataset, chọn **Add Input** rồi **Run All**. Notebook tìm file trong `/kaggle/input` và hỗ trợ chạy cục bộ từ project.

Nhãn học là `Fire_Alarm_Transient`. Nhãn 1 biểu thị pha khởi phát PM đã chú giải. Nhãn 0 biểu thị ngoài cửa sổ khởi phát, bao gồm các giai đoạn sự kiện đang tiếp diễn. Nhãn hiện được suy ra từ cảm biến, chưa xác nhận bằng nhật ký cháy vật lý.

Notebook giữ đúng 13 đặc trưng. Chấm điểm bằng ROC AUC, Average Precision, log loss và Brier score. Điểm đầu ra mang nghĩa xác suất lớp khởi phát theo nhãn này. Tầng ngưỡng cảnh báo và biểu quyết được cấu hình ở hệ thống sử dụng model.

Validation luân phiên ba nhóm A/B/C, giữ các cặp phiên gần trùng cùng phía. Đây là đánh giá phát triển trên ba nhóm, chưa có tập test thực địa độc lập. Model cuối được fit lại trên toàn bộ các hàng đủ điều kiện sau khi hoàn thành validation.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import zipfile

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = None
    display = print

# Set a specific path here if the same filename occurs in several Kaggle Inputs.
DATASET_PATH = None
OUTPUT_DIR = Path('/kaggle/working/lightgbm13_transient_bundle') if Path('/kaggle/working').exists() else Path('lightgbm13_transient_bundle')
TARGET = 'Fire_Alarm_Transient'
FEATURES = [
    'd_temp_1', 'd_temp_5', 'd_hum_1', 'd_hum_5',
    'd_pm1_1', 'd_pm1_5', 'd_pm25_1', 'd_pm25_5',
    'temp_roll_5_std', 'hum_roll_5_std', 'pm1_roll_5_std',
    'pm25_roll_5_std', 'pm_ratio',
]
SENSORS = {
    'Temperature[C]': 'temp', 'Humidity[%]': 'hum',
    'PM1.0': 'pm1', 'PM2.5': 'pm25',
}
PARAMETERS = {
    'n_estimators': 150, 'max_depth': 6, 'num_leaves': 31,
    'learning_rate': 0.05, 'random_state': 42,
    'n_jobs': 2, 'verbosity': -1,
}
versions = {
    'python': platform.python_version(), 'lightgbm': lgb.__version__,
    'pandas': pd.__version__, 'numpy': np.__version__,
    'scikit_learn': sklearn.__version__, 'joblib': joblib.__version__,
}
print(json.dumps(versions, indent=2))


{
  "python": "3.12.16",
  "lightgbm": "4.6.0",
  "pandas": "2.2.3",
  "numpy": "2.2.6",
  "scikit_learn": "1.6.1",
  "joblib": "1.4.2"
}


## Nạp và kiểm tra dữ liệu

Các cột chú giải chỉ phục vụ chọn hàng, ghép kết quả và chia nhóm. Chỉ danh sách `FEATURES` được đưa vào LightGBM. Bộ dữ liệu v1 dự kiến có 62.630 hàng và 832 mẫu đích dương.

In [2]:
filename = 'smoke_detection_iot_transient_v1.csv'
if DATASET_PATH is None:
    candidates = []
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(filename))
    candidates.extend([
        Path.cwd() / filename,
        Path.cwd() / 'Model/smoke_transient_v1' / filename,
    ])
    candidates = sorted({path.resolve() for path in candidates if path.is_file()})
    if len(candidates) != 1:
        raise ValueError(f'Expected one dataset. Set DATASET_PATH explicitly. Found {candidates}')
    dataset_path = candidates[0]
else:
    dataset_path = Path(DATASET_PATH)

dataset_sha256 = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
df = pd.read_csv(dataset_path, float_precision='round_trip')
required = [*SENSORS, 'UTC', 'CNT', 'session_id', 'source_group', TARGET, 'training_eligible', 'label_version']
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f'Missing columns: {missing}')
if not df[TARGET].isin([0, 1]).all():
    raise ValueError('Target must contain binary 0/1 labels')
if not np.isfinite(df[[*SENSORS, 'UTC', 'CNT', 'session_id']].to_numpy(dtype=float)).all():
    raise ValueError('Invalid raw measurements or timestamps')

def read_boolean(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    converted = series.astype(str).str.strip().str.lower().map({
        'true': True, 'false': False, '1': True, '0': False,
    })
    if converted.isna().any():
        raise ValueError(f'Invalid boolean values in {series.name}')
    return converted.astype(bool)

df['training_eligible'] = read_boolean(df['training_eligible'])
if 'needs_review' in df:
    df['needs_review'] = read_boolean(df['needs_review'])
    if (df['needs_review'] & df['training_eligible']).any():
        raise ValueError('Rows requiring annotation review were marked eligible')
expected_groups = {1: 'A', 5: 'A', 2: 'B', 3: 'B', 4: 'C'}
if not df.session_id.map(expected_groups).eq(df.source_group).all():
    raise ValueError('Source grouping differs from the reviewed five-session dataset')
if df.duplicated(['session_id', 'CNT']).any():
    raise ValueError('Duplicate sample keys')
if df.label_version.nunique() != 1:
    raise ValueError('Mixed annotation versions')

df = df.sort_values(['session_id', 'UTC', 'CNT']).reset_index(drop=True)
print('Dataset:', dataset_path)
print('SHA-256:', dataset_sha256)
print('Raw shape:', df.shape, '| target positives:', int(df[TARGET].sum()))


Dataset: /audit/Model/smoke_transient_v1/smoke_detection_iot_transient_v1.csv
SHA-256: 141056bbdd26ad78b0c4cec63ca665152d04a4f70fa95095d09d29c590a98159
Raw shape: (62630, 30) | target positives: 832


## Tính đúng 13 đặc trưng trước khi lọc hàng

Mỗi hàng dùng hiện tại và lịch sử. Chênh lệch 5 mẫu cần sáu mẫu liên tục. Cửa sổ độ lệch chuẩn dùng tối đa năm mẫu, `ddof=1`, cùng công thức backend. Reset lịch sử khi đổi phiên hoặc UTC/CNT không liên tục.

Sau khi tính trên toàn chuỗi, mới chọn `training_eligible=True` và đủ lịch sử. Các hàng bị loại khỏi loss vẫn giữ vai trò lịch sử cảm biến nếu chuỗi đo liên tục.

In [3]:
reset = (
    df.session_id.ne(df.session_id.shift())
    | df.UTC.diff().ne(1)
    | df.CNT.diff().ne(1)
)
block = reset.cumsum()
history_ready = df.groupby(block, sort=False).cumcount().ge(5)
X_full = pd.DataFrame(index=df.index)
for sensor, prefix in SENSORS.items():
    groups = df[sensor].groupby(block, sort=False)
    X_full[f'd_{prefix}_1'] = groups.diff(1).fillna(0.0)
    X_full[f'd_{prefix}_5'] = groups.diff(5).fillna(0.0)
    X_full[f'{prefix}_roll_5_std'] = groups.transform(
        lambda part: part.rolling(5, min_periods=1).std(ddof=1).fillna(0.0)
    )
X_full['pm_ratio'] = (df['PM2.5'] + 1e-3) / (df['PM1.0'] + 1e-3)
X_full = X_full.loc[:, FEATURES]
assert X_full.shape[1] == 13
if not np.isfinite(X_full.to_numpy()).all():
    raise ValueError('Nonfinite engineered features')

eligible = df.training_eligible & history_ready
X = X_full.loc[eligible].copy()
y = df.loc[eligible, TARGET].astype(int)
groups = df.loc[eligible, 'source_group']
group_table = df.loc[eligible].groupby('source_group')[TARGET].agg(rows='size', positive='sum')
group_table['negative'] = group_table['rows'] - group_table['positive']
if (group_table[['positive', 'negative']] == 0).any().any():
    raise ValueError('A validation group lacks one class')
print('Eligible features:', X.shape)
display(group_table)
display(X.head())


Eligible features: (62482, 13)
               rows  positive  negative
source_group                           
A             11478       658     10820
B             49896       108     49788
C              1108        66      1042
   d_temp_1  d_temp_5  d_hum_1  ...  pm1_roll_5_std  pm25_roll_5_std  pm_ratio
5     -0.05     -0.23     0.11  ...        0.052631         0.055498  1.039284
6      0.00     -0.19     0.04  ...        0.063008         0.064265  1.038119
7     -0.02     -0.16     0.07  ...        0.071204         0.073824  1.037329
8     -0.04     -0.16     0.07  ...        0.076616         0.076616  1.036870
9     -0.01     -0.12     0.10  ...        0.061887         0.063797  1.041135

[5 rows x 13 columns]


## Validation theo nhóm và đánh giá điểm

Ba lượt giữ lần lượt nhóm A, B, C ngoài train. Mỗi hàng validation được chấm bởi model chưa học nhóm đó. Siêu tham số cố định trong cả ba lượt. Báo riêng điểm train/validation để thấy mức chênh lệch.

ROC AUC và Average Precision càng cao càng tốt. Log loss và Brier score càng thấp càng tốt. Average Precision cần đối chiếu với tỷ lệ dương của chính nhóm đó. OOF là các điểm ngoài fold từ ba model khác nhau; chỉ số gộp OOF được dùng bổ sung cho bảng từng nhóm.

In [4]:
def score_metrics(truth, probability):
    return {
        'rows': int(len(truth)), 'positive': int(np.sum(truth)),
        'positive_rate': float(np.mean(truth)),
        'roc_auc': float(roc_auc_score(truth, probability)),
        'average_precision': float(average_precision_score(truth, probability)),
        'log_loss': float(log_loss(truth, probability, labels=[0, 1])),
        'brier_score': float(brier_score_loss(truth, probability)),
    }

splitter = LeaveOneGroupOut()
oof_probability = pd.Series(np.nan, index=X.index, dtype=float)
fold_assignment = pd.Series('', index=X.index, dtype=object)
metric_rows = []
for train_positions, val_positions in splitter.split(X, y, groups):
    train_index = X.index[train_positions]
    val_index = X.index[val_positions]
    held_group = str(groups.loc[val_index].iloc[0])
    assert set(groups.loc[train_index]).isdisjoint(set(groups.loc[val_index]))
    model = lgb.LGBMClassifier(**PARAMETERS)
    model.fit(X.loc[train_index], y.loc[train_index])
    positive_index = list(model.classes_).index(1)
    train_probability = model.predict_proba(X.loc[train_index])[:, positive_index]
    val_probability = model.predict_proba(X.loc[val_index])[:, positive_index]
    oof_probability.loc[val_index] = val_probability
    fold_assignment.loc[val_index] = held_group
    for subset, truth, probability in [
        ('train', y.loc[train_index], train_probability),
        ('validation', y.loc[val_index], val_probability),
    ]:
        metric_rows.append({'held_out_group': held_group, 'subset': subset,
                            **score_metrics(truth, probability)})
    print(f'Completed held-out group {held_group}')

assert oof_probability.notna().all()
fold_metrics = pd.DataFrame(metric_rows)
oof_metrics = score_metrics(y, oof_probability)
display(fold_metrics.round(6))
print('Supplementary pooled OOF metrics:')
display(pd.DataFrame([oof_metrics]).round(6))

validation_metrics = fold_metrics.loc[fold_metrics['subset'].eq('validation')]
print('Unweighted mean of three group validation results:')
display(validation_metrics[['roc_auc', 'average_precision', 'log_loss', 'brier_score']].mean().to_frame('mean'))


Completed held-out group A
Completed held-out group B
Completed held-out group C
  held_out_group      subset   rows  ...  average_precision  log_loss  brier_score
0              A       train  51004  ...           1.000000  0.000545     0.000061
1              A  validation  11478  ...           0.277139  0.411804     0.054853
2              B       train  12586  ...           0.997430  0.018931     0.004282
3              B  validation  49896  ...           0.182378  0.014845     0.001974
4              C       train  61374  ...           0.999997  0.001423     0.000198
5              C  validation   1108  ...           0.127943  0.888166     0.220614

[6 rows x 9 columns]
Supplementary pooled OOF metrics:
    rows  positive  positive_rate  ...  average_precision  log_loss  brier_score
0  62482       832       0.013316  ...           0.166314  0.103254     0.015565

[1 rows x 7 columns]
Unweighted mean of three group validation results:
                       mean
roc_auc            

## Fit model cuối và xuất bundle chấm điểm

Model cuối sử dụng toàn bộ các hàng đủ điều kiện. Chỉ số train của model này được ghi riêng. Các điểm OOF vẫn đến từ ba model validation phía trên.

Bundle gồm model, metadata về mục tiêu và 13 đặc trưng, chỉ số từng fold và điểm OOF từng hàng. Metadata dành cho model khởi phát, cần được hệ thống suy luận đọc đúng ý nghĩa nhãn khi tích hợp.

In [5]:
final_model = lgb.LGBMClassifier(**PARAMETERS)
final_model.fit(X, y)
positive_index = list(final_model.classes_).index(1)
final_train_probability = final_model.predict_proba(X)[:, positive_index]
final_train_metrics = score_metrics(y, final_train_probability)
print('Final model training metrics:')
display(pd.DataFrame([final_train_metrics]).round(6))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, OUTPUT_DIR / 'lightgbm13_transient_model.joblib')
metric_payload = {
    'folds': metric_rows, 'pooled_oof': oof_metrics,
    'final_model_train': final_train_metrics,
}
metadata = {
    'model_name': 'LightGBM13_Transient_Onset_v1',
    'target': TARGET, 'positive_label': 1, 'classes': [int(v) for v in final_model.classes_],
    'features': FEATURES, 'n_features': len(FEATURES),
    'label_version': str(df.label_version.iloc[0]),
    'positive_meaning': 'Sensor-inferred PM onset window',
    'negative_meaning': 'Outside onset, including continuation and recovery',
    'physical_fire_ground_truth': 'unverified',
    'probability_calibration': 'not independently validated',
    'feature_extraction': {
        'history_samples': 6, 'rolling_window': 5, 'std_ddof': 1,
        'pm_ratio_epsilon': 1e-3, 'expected_sampling_seconds': 1,
        'reset_on_session_or_sample_gap': True,
    },
    'validation': {
        'method': 'LeaveOneGroupOut',
        'source_groups': {'A': [1, 5], 'B': [2, 3], 'C': [4]},
        'independent_field_test_available': False,
        'per_group_metrics_file': 'metrics.json',
    },
    'source': {'filename': dataset_path.name, 'sha256': dataset_sha256,
               'raw_rows': len(df), 'training_rows': len(X), 'training_positives': int(y.sum())},
    'parameters': PARAMETERS, 'versions': versions,
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metric_payload, ensure_ascii=False, indent=2))

key_columns = [column for column in [
    'source_row_position', 'session_id', 'source_group', 'UTC', 'CNT',
    'transient_event_id', 'boundary_uncertain', TARGET,
] if column in df.columns]
oof_table = df.loc[X.index, key_columns].copy()
oof_table['held_out_group'] = fold_assignment
oof_table['oof_probability_onset'] = oof_probability
oof_table.to_csv(OUTPUT_DIR / 'oof_scores.csv', index=False)
fold_metrics.to_csv(OUTPUT_DIR / 'fold_metrics.csv', index=False)

# Reload the exported estimator and check class/feature order and scoring parity.
restored = joblib.load(OUTPUT_DIR / 'lightgbm13_transient_model.joblib')
assert list(restored.feature_name_) == FEATURES
probe = X.iloc[:min(100, len(X))]
np.testing.assert_allclose(restored.predict_proba(probe), final_model.predict_proba(probe))
assert set(oof_table.held_out_group) == set(groups)

zip_path = OUTPUT_DIR.parent / 'lightgbm13_transient_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)
print('Bundle:', zip_path)
if FileLink is not None:
    display(FileLink(str(zip_path)))


Final model training metrics:
    rows  positive  positive_rate  ...  average_precision  log_loss  brier_score
0  62482       832       0.013316  ...           0.994866  0.005071     0.001156

[1 rows x 7 columns]
Bundle: /audit/output/analysis/lightgbm13_transient_trial/lightgbm13_transient_bundle.zip
